In [ ]:
from bigmodule import M, I

# <aistudiograph>

# @param(id="m9", name="run")
def m9_run_bigquant_run(input_1, input_2, input_3):
    # Python 代码入口函数，input_1/2/3 对应三个输入端，data_1/2/3 对应三个输出端
    # 示例代码如下。在这里编写您的代码
    import dai

    df = input_1.read()
    
    drop_cols =['st_status', 'low', 'low_1', 'price_limit_status']
    
    df = df.drop(columns=drop_cols)

   
    ds = dai.DataSource.write_bdb(df)

    return dict(data_1=ds, data_2={"hello": "world"}, data_3=None)

# @param(id="m9", name="post_run_outputs_")
def m9_post_run_outputs__bigquant_run(outputs):
    # 后处理函数，可选。输入是主函数的输出，可以在这里对数据做处理，或者返回更友好的outputs数据格式。此函数输出不会被缓存。
    return outputs

# @param(id="m19", name="initialize")
# 交易引擎：初始化函数，只执行一次
def m19_initialize_bigquant_run(context):
    from bigtrader.finance.commission import PerOrder

    # 系统已经设置了默认的交易手续费和滑点，要修改手续费可使用如下函数
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))
# @param(id="m19", name="before_trading_start")
# 交易引擎：每个单位时间开盘前调用一次。
def m19_before_trading_start_bigquant_run(context, data):
    # 盘前处理，订阅行情等
    pass

# @param(id="m19", name="handle_tick")
# 交易引擎：tick数据处理函数，每个tick执行一次
def m19_handle_tick_bigquant_run(context, tick):
    pass

# @param(id="m19", name="handle_data")
def m19_handle_data_bigquant_run(context, data):
    import pandas as pd

    # 下一个交易日不是调仓日，则不生成信号
    if not context.rebalance_period.is_signal_date(data.current_dt.date()):
        return

    # 从传入的数据 context.data 中读取今天的信号数据
    today_df = context.data[context.data["date"] == data.current_dt.strftime("%Y-%m-%d")]

    # 卖出不在目标持有列表中的股票
    for instrument in sorted(set(context.get_account_positions().keys()) - set(today_df["instrument"])):
        context.order_target_percent(instrument, 0)
    # 买入目标持有列表中的股票
    for i, x in today_df.iterrows():
        context.order_target_percent(x.instrument, 0.0 if pd.isnull(x.position) else x.position)
# @param(id="m19", name="handle_trade")
# 交易引擎：成交回报处理函数，每个成交发生时执行一次
def m19_handle_trade_bigquant_run(context, trade):
    pass

# @param(id="m19", name="handle_order")
# 交易引擎：委托回报处理函数，每个委托变化时执行一次
def m19_handle_order_bigquant_run(context, order):
    pass

# @param(id="m19", name="after_trading")
# 交易引擎：盘后处理函数，每日盘后执行一次
def m19_after_trading_bigquant_run(context, data):
    pass

# @module(position="-784,-1291", comment="""训练集""")
m1 = M.instruments_dai.v3(
    start_date="""2021-01-01""",
    start_date_bound_to_trading_date=False,
    end_date="""2024-12-31""",
    end_date_bound_to_trading_date=False,
    market="""中国A股""",
    custom_market="""""",
    instrument_list="""""",
    m_name="""m1"""
)

# @module(position="-929,-1165", comment="""""", comment_collapsed=True)
m4 = M.auto_labeler.v2(
    instruments=m1.data,
    features_expr="""m_lead(close, 2) / m_lead(open, 1) AS _future_return
all_quantile_cont(_future_return, 0.01) AS _future_return_1pct
all_quantile_cont(_future_return, 0.99) AS _future_return_99pct
clip(_future_return, _future_return_1pct, _future_return_99pct) AS _clipped_return
all_cbins(_clipped_return, 70) AS _binned_return
_binned_return AS _label
if( m_lead(high, 1) = m_lead(low, 1), NULL, _label) as label
""",
    start_date="""""",
    end_date="""""",
    before_start_days=90,
    m_name="""m4"""
)

# @module(position="-405,-1397", comment="""""", comment_collapsed=True)
m2 = M.input_features_expr.v1(
    features="""pe_ttm
pb
ps_ttm
pcf_net_leading
pcf_op_ttm
total_market_cap
float_market_cap
dividend_yield_ratio
-- 5日动量/反转
momentum_5
reversal_5
-- 5日波动
volatility_5
-- 换手
turn


# 过滤因子，这四个因子不参与模型学习
st_status 
low   
price_limit_status
m_lag(low,1) as low_1
""",
    m_name="""m2"""
)

# @module(position="-586,-1167", comment="""""", comment_collapsed=True)
m3 = M.extract_data_expr.v2(
    instruments=m1.data,
    features=m2.data,
    start_date="""""",
    end_date="""""",
    before_start_days=90,
    m_name="""m3"""
)

# @module(position="-746,-1072", comment="""""", comment_collapsed=True)
m8 = M.data_join.v4(
    data1=m4.data,
    data2=m3.data,
    on="""date,instrument""",
    how="""inner""",
    sort=True,
    m_name="""m8"""
)

# @module(position="-745,-985", comment="""""", comment_collapsed=True)
m6 = M.data_filter.v5(
    input_data=m8.data,
    expr="""st_status==0 & price_limit_status==2 | price_limit_status ==3 & low>low_1""",
    output_left_data=False,
    m_name="""m6"""
)

# @module(position="-738,-910", comment="""""", comment_collapsed=True)
m12 = M.data_dropnan.v6(
    input_data=m6.data,
    m_name="""m12"""
)

# @module(position="-739,-829", comment="""""", comment_collapsed=True)
m9 = M.python.v2(
    input_1=m12.data,
    run=m9_run_bigquant_run,
    do_run=True,
    post_run_outputs_=m9_post_run_outputs__bigquant_run,
    m_name="""m9"""
)

# @module(position="-384,-767", comment="""""", comment_collapsed=True)
m7 = M.stock_ranker_dai_train.v9(
    data=m9.data_1,
    learning_algorithm="""排序""",
    number_of_leaves=32,
    min_docs_per_leaf=1000,
    number_of_trees=20,
    learning_rate=0.1,
    max_bins=1072,
    feature_fraction=1,
    data_row_fraction=1,
    plot_charts=True,
    ndcg_discount_base=1,
    m_name="""m7"""
)

# @module(position="6,-1233", comment="""测试集""")
m15 = M.instruments_dai.v3(
    start_date="""2025-01-01""",
    start_date_bound_to_trading_date=False,
    end_date="""2025-12-03""",
    end_date_bound_to_trading_date=True,
    market="""中国A股""",
    custom_market="""""",
    instrument_list="""""",
    m_cached=False,
    m_name="""m15"""
)

# @module(position="10,-1115", comment="""""", comment_collapsed=True)
m16 = M.extract_data_expr.v2(
    instruments=m15.data,
    features=m2.data,
    start_date="""""",
    end_date="""""",
    before_start_days=90,
    m_name="""m16"""
)

# @module(position="12,-998", comment="""""", comment_collapsed=True)
m10 = M.data_filter.v5(
    input_data=m16.data,
    expr="""st_status==0 & price_limit_status==2 | price_limit_status ==3 & low>low_1""",
    output_left_data=False,
    m_name="""m10"""
)

# @module(position="13,-902", comment="""""", comment_collapsed=True)
m17 = M.data_dropnan.v6(
    input_data=m10.data,
    m_name="""m17"""
)

# @module(position="-384,-698", comment="""""", comment_collapsed=True)
m13 = M.stock_ranker_dai_predict.v13(
    model=m7.model,
    data=m17.data,
    m_name="""m13"""
)

# @module(position="-385,-619", comment="""等权分配""", comment_collapsed=True)
m18 = M.score_to_position.v4(
    input_1=m13.predictions,
    score_field="""score DESC""",
    hold_count=10,
    position_expr="""-- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
-- 在这里输入表达式, 每行一个表达式, 输出仓位字段必须命名为 position, 模块会进一步做归一化
-- 排序倒数: 1 / score_rank AS position
-- 对数下降: 1 / log2(score_rank + 1) AS position
-- TODO 拟合、最优化 ..

-- 等权重分配
1 AS position
""",
    total_position=1,
    extract_data=True,
    m_name="""m18"""
)

# @module(position="-385,-542", comment="""交易，日线，设置初始化函数和K线处理函数，以及初始化资金、基准等""", comment_collapsed=True)
m19 = M.bigtrader.v47(
    data=m18.data,
    start_date="""""",
    end_date="""""",
    initialize=m19_initialize_bigquant_run,
    before_trading_start=m19_before_trading_start_bigquant_run,
    handle_tick=m19_handle_tick_bigquant_run,
    handle_data=m19_handle_data_bigquant_run,
    handle_trade=m19_handle_trade_bigquant_run,
    handle_order=m19_handle_order_bigquant_run,
    after_trading=m19_after_trading_bigquant_run,
    capital_base=1000000,
    frequency="""daily""",
    product_type="""股票""",
    rebalance_period_type="""交易日""",
    rebalance_period_days="""2""",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="""标准模式""",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="""open""",
    order_price_field_sell="""open""",
    benchmark="""沪深300指数""",
    plot_charts=True,
    debug=False,
    backtest_only=False,
    m_name="""m19"""
)
# </aistudiograph>

[2025-12-04 14:04:23] [info     ] instruments_dai.v3 开始运行 ..
[2025-12-04 14:04:23] [info     ] instruments_dai.v3 命中缓存
[2025-12-04 14:04:23] [info     ] instruments_dai.v3 运行完成 [0.023s].
[2025-12-04 14:04:23] [info     ] auto_labeler.v2 开始运行 ..
[2025-12-04 14:04:23] [info     ] auto_labeler.v2 命中缓存
[2025-12-04 14:04:23] [info     ] auto_labeler.v2 运行完成 [0.027s].
[2025-12-04 14:04:23] [info     ] input_features_expr.v1 开始运行 ..
[2025-12-04 14:04:23] [info     ] input_features_expr.v1 命中缓存
[2025-12-04 14:04:23] [info     ] input_features_expr.v1 运行完成 [0.025s].
[2025-12-04 14:04:23] [info     ] extract_data_expr.v2 开始运行 ..
[2025-12-04 14:04:23] [info     ] extract_data_expr.v2 命中缓存
[2025-12-04 14:04:23] [info     ] extract_data_expr.v2 运行完成 [0.022s].
[2025-12-04 14:04:23] [info     ] data_join.v4 开始运行 ..
[2025-12-04 14:04:23] [info     ] data_join.v4 命中缓存
[2025-12-04 14:04:23] [info     ] data_join.v4 运行完成 [0.023s].
[2025-12-04 14:04:23] [info     ] data_filter.v5 开始运行 ..
[2025-12-04 14:04

[2025-12-04 14:04:23] [info     ] stock_ranker_dai_train.v9 运行完成 [0.101s].
[2025-12-04 14:04:23] [info     ] instruments_dai.v3 开始运行 ..
[2025-12-04 14:04:24] [info     ] instruments_dai.v3 运行完成 [0.047s].
[2025-12-04 14:04:24] [info     ] extract_data_expr.v2 开始运行 ..
[2025-12-04 14:04:24] [info     ] start_date='2025-01-01', end_date='2025-12-03', query_start_date='2024-10-03' (支持加速 [url="command:switch-quota"]升级资源[/url]) ..
